# Solution A1 · A KNN in numpy: vectorized distances, voting and weights

**Type**: Implement from scratch

## Objective

Implement a KNN classifier without loops over the examples: distance matrix (with the identity $\lVert a-b\rVert^2 = \lVert a\rVert^2+\lVert b\rVert^2-2a\cdot b$),
selection of the `k` neighbors, uniform voting and voting weighted by the inverse of the distance, and handling of zero distance. Check that the predictions match `KNeighborsClassifier`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)
print(Z_tr.shape, Z_te.shape, "| classes:", np.bincount(y_tr))

## Commented solution

In [ ]:
def distances(A, B):
    square = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(square, 0))              # rounding can give small negatives


def nearest(D, k):
    idx = np.argpartition(D, k - 1, axis=1)[:, :k]                        # the k smallest, unsorted (linear cost)
    order = np.argsort(np.take_along_axis(D, idx, 1), axis=1)             # I sort only these k
    return np.take_along_axis(idx, order, 1)

In [ ]:
def predict_knn(A, X, y, k, weights="uniform"):
    D = distances(A, X)
    idx = nearest(D, k)
    d = np.take_along_axis(D, idx, 1)
    n_classes = y.max() + 1
    predictions = np.empty(len(A), dtype=int)
    for i in range(len(A)):
        if weights == "uniform":
            w = np.ones(k)
        elif (d[i] == 0).any():
            w = (d[i] == 0).astype(float)                                # only those that coincide with the point count
        else:
            w = 1 / d[i]
        predictions[i] = np.bincount(y[idx[i]], weights=w, minlength=n_classes).argmax()     # on a tie: smallest label
    return predictions

In [ ]:
agreement = {}
for k in (1, 5, 15):
    for weights in ("uniform", "distance"):
        sk = KNeighborsClassifier(k, weights=weights).fit(Z_tr, y_tr).predict(Z_te)
        agreement[(k, weights)] = float((predict_knn(Z_te, Z_tr, y_tr, k, weights) == sk).mean())
print(agreement)

## Check

In [ ]:
with verify():
    check(nearest(np.array([[3.0, 1.0, 2.0, 5.0]]), 2).tolist() == [[1, 2]], "nearest returns the k smallest indices, in increasing order", "argpartition then sort only the k found")
    D = distances(Z_te, Z_tr)
    check(D.shape == (len(Z_te), len(Z_tr)) and nearest(D, 5).shape == (len(Z_te), 5), "correct shapes", "distances: (len(A), len(B))")
    check(nearest(np.array([[0.0, 0.0, 0.0]]), 1).shape == (1, 1), "nearest also works with equal distances", "mind the ties")
    check(nearest(D, 5).tolist() == np.argsort(D, axis=1)[:, :5].tolist(), "the indices of the neighbors match a full sort", "argpartition(D, k - 1)[:, :k] then sort by distance")
    check(np.allclose(D, pairwise_distances(Z_te, Z_tr), atol=1e-6) and not np.isnan(distances(Z_tr, Z_tr)).any(),
          "distances equal to pairwise_distances, with no NaN even between identical points", "np.maximum(..., 0) before the root")
    for (k, weights), fraction in agreement.items():
        print(f"   k={k:2d} weights={weights:8s} agreement with scikit-learn: {fraction:.3f}")
    check(set(agreement) == {(k, p) for k in (1, 5, 15) for p in ("uniform", "distance")} and all(v == 1.0 for v in agreement.values()), "all the predictions match scikit-learn", "check the ties (smallest label) and the weights 1/d")
    check((predict_knn(Z_tr, Z_tr, y_tr, 3, "distance") == y_tr).all(), "with 1/distance weights every training point predicts itself (distance 0)", "if there is a neighbor at distance 0 only those at distance 0 count")
done()

## What you learned

- A KNN's `predict` is three lines of algebra: the distance matrix (with the dot-product identity, which exploits numpy's optimized matrix multiplication), the indices of the `k` minima with
  `argpartition` (linear cost, much better than sorting everything) and a vote. The model is the training set.
- **Two numeric details**: the square computed with the identity can come out as `-1e-16` and the root gives `NaN`, which is why it is brought to zero with `np.maximum`; and with `1/distance` weights a neighbor at distance 0 would cause a
  division by zero, and scikit-learn resolves it by giving weight only to the neighbors at distance 0.
- The tie is resolved **deterministically but arbitrarily** (smallest label): two identical models with renamed labels can give different answers. With odd `k` and two classes there are no ties.
- Cost: the distance matrix is `n_test × n_train`; with hundreds of thousands of rows it no longer fits in memory and you work in blocks: it is what `KNeighborsClassifier` and the approximate search libraries do.

## Common pitfall

Computing the distances with two `for` loops "for clarity": it works with 178 wines and becomes unusable with 100,000 rows. The vectorized version is not an optional optimization, it is what makes KNN usable.

## Interview question

<details><summary>What is the complexity of `predict` for a KNN and how is it reduced?</summary>

For every test point it is O(n·d) with the exhaustive computation (n training examples, d features), plus O(n) to select the k minima with a partition. It is reduced with search structures (kd-tree, ball-tree: good with few dimensions)
or with approximate indexes (LSH, HNSW) in high dimension, at the cost of some errors in the neighbors found. In production you also precompute the scalers and keep the training set in memory.

</details>